# 12 — Failure Boundary Phase-Lock

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

Notebook 12 intentionally pushes the control stack toward the **CGCS 45° phase-lock boundary**.

Previous notebooks mostly stayed in the high-stability regime:

```text
cos θ ≈ 1
```

This notebook asks:

> What happens when drift, noise, and model mismatch push response similarity toward the 45° threshold?

---

## Goal

Stress-test controllers under boundary pressure.

Compare:

- none
- moving average
- scalar Kalman
- joint Kalman
- adaptive gated Kalman
- constrained MPC-lite
- oracle

---

## Boundary condition

CGCS phase-lock threshold:

```text
cos θ ≥ 1 / √(1² + 1²) ≈ 0.7071
```

A policy fails if:

```text
cos θ < 0.7071
```


## Failure modes tested

Notebook 12 combines:

1. **Large Ω drift**
2. **B offset drift**
3. **measurement burst**
4. **model mismatch**
5. **late-stage coupling shock**

This is deliberately harder than Notebooks 08–11.


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures/failure_boundary_phase_lock"
RESULTS_DIR = "results"
DOCS_DIR = "docs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

def save_fig(name):
    plt.savefig(f"{FIG_DIR}/12_failure_boundary_phase_lock_{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/12_failure_boundary_phase_lock_{name}.pdf", bbox_inches="tight")


In [ ]:
def rabi_model(t, A, Omega, B):
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def distorted_rabi_model(t, A, Omega, B, distortion=0.0):
    base = rabi_model(t, A, Omega, B)
    # small unmodeled harmonic distortion
    return np.clip(base + distortion * np.sin(1.7 * t + 0.4) * np.sin(0.5 * Omega * t), 0, 1)


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def phase_lock_metric(signal, target):
    dot = np.dot(signal, target)
    norm = np.linalg.norm(signal) * np.linalg.norm(target)
    if norm == 0:
        return np.nan
    return dot / norm


def moving_average(x, window=4):
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)
    innovation = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        innov = measurement - x_pred
        innovation[i] = innov

        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * innov
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist, innovation


def joint_kalman(z, q_omega, q_b, q_cross, r_omega, r_b):
    z = np.asarray(z, dtype=float)
    n = z.shape[0]

    F = np.eye(2)
    H = np.eye(2)
    Q = np.array([[q_omega, q_cross], [q_cross, q_b]])
    R = np.array([[r_omega, 0.0], [0.0, r_b]])

    x = z[0].reshape(2, 1)
    P = np.eye(2)

    out = np.zeros((n, 2))
    innovations = np.zeros((n, 2))

    for i in range(n):
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        y = z[i].reshape(2, 1) - H @ x_pred
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)

        x = x_pred + K @ y
        P = (np.eye(2) - K @ H) @ P_pred

        out[i] = x.ravel()
        innovations[i] = y.ravel()

    return out, innovations


def robust_gated_joint_kalman(z, q_omega, q_b, q_cross, r_omega, r_b, gate_omega=0.12, gate_b=0.035):
    z = np.asarray(z, dtype=float)
    n = z.shape[0]

    F = np.eye(2)
    H = np.eye(2)
    Q = np.array([[q_omega, q_cross], [q_cross, q_b]])
    R_base = np.array([[r_omega, 0.0], [0.0, r_b]])

    x = z[0].reshape(2, 1)
    P = np.eye(2)

    out = np.zeros((n, 2))
    innovations = np.zeros((n, 2))
    gated = np.zeros(n, dtype=bool)

    for i in range(n):
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        y = z[i].reshape(2, 1) - H @ x_pred
        innov = y.ravel()
        innovations[i] = innov

        R = R_base.copy()
        if abs(innov[0]) > gate_omega or abs(innov[1]) > gate_b:
            # Inflate R to down-weight sudden inconsistent observations.
            R *= 40.0
            gated[i] = True

        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)

        x = x_pred + K @ y
        P = (np.eye(2) - K @ H) @ P_pred

        out[i] = x.ravel()

    return out, innovations, gated


def bounded_command(command, bound):
    command = np.asarray(command)
    bounded = np.zeros_like(command)
    bounded[0] = np.clip(command[0], -bound, bound)

    for i in range(1, len(command)):
        step = np.clip(command[i] - bounded[i - 1], -bound, bound)
        bounded[i] = bounded[i - 1] + step

    return bounded


def constrained_mpc_command(signal, delta_bound, smooth=0.65):
    raw = np.asarray(signal)
    bounded = bounded_command(raw, delta_bound)
    out = np.zeros_like(bounded)
    out[0] = bounded[0]
    for i in range(1, len(out)):
        out[i] = smooth * out[i - 1] + (1 - smooth) * bounded[i]
    return out


def evaluate_policy(name, delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    omega_controlled = target_Omega + true_delta_omega - delta_omega_hat
    b_controlled = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.45)

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    response_rmse = []
    phase_lock = []
    controlled_signals = []

    for k in range(len(true_delta_omega)):
        y = distorted_rabi_model(
            pulse_t,
            target_A,
            omega_controlled[k],
            b_controlled[k],
            distortion=distortion_profile[k] * residual_distortion_scale,
        )
        controlled_signals.append(y)
        response_rmse.append(rmse(y, target_signal_local))
        phase_lock.append(phase_lock_metric(y, target_signal_local))

    response_rmse = np.array(response_rmse)
    phase_lock = np.array(phase_lock)
    controlled_signals = np.array(controlled_signals)

    threshold = 1 / np.sqrt(2)
    failures = phase_lock < threshold

    if np.any(failures):
        first_failure = int(np.where(failures)[0][0])
    else:
        first_failure = None

    return {
        "name": name,
        "omega_command": np.asarray(delta_omega_hat),
        "b_command": np.asarray(delta_b_hat),
        "omega_rmse": rmse(omega_controlled, np.full(len(omega_controlled), target_Omega)),
        "b_rmse": rmse(b_controlled, np.full(len(b_controlled), target_B)),
        "response_rmse": response_rmse,
        "response_rmse_mean": float(np.mean(response_rmse)),
        "response_rmse_max": float(np.max(response_rmse)),
        "phase_lock": phase_lock,
        "min_phase_lock": float(np.min(phase_lock)),
        "mean_phase_lock": float(np.mean(phase_lock)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(phase_lock, -1, 1))))),
        "all_blocks_phase_locked": bool(np.all(phase_lock >= threshold)),
        "failure_count": int(np.sum(failures)),
        "first_failure_block": first_failure,
        "controlled_signals": controlled_signals,
    }


## Simulate boundary-pressure environment


In [ ]:
n_blocks = 80
block_times = np.arange(n_blocks)
pulse_t = np.linspace(0, 10, 140)

target_A = 0.86
target_Omega = 2.45
target_B = 0.06
target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

# Strong drift terms designed to push no-control toward failure boundary.
slow = 0.42 * np.sin(2 * np.pi * block_times / 31 + 0.1)
fast = 0.16 * np.sin(2 * np.pi * block_times / 9)
ramp = 0.0035 * (block_times - n_blocks / 2)

true_delta_Omega = slow + fast + ramp

shared = 0.055 * np.sin(2 * np.pi * block_times / 27 + 0.6)
b_trend = 0.0010 * block_times
true_delta_B = b_trend + 0.020 * np.sin(2 * np.pi * block_times / 18) + 0.22 * shared

# Coupling shock near the end.
shock_start, shock_end = 54, 68
true_delta_Omega[shock_start:shock_end] += 0.23
true_delta_B[shock_start:shock_end] += 0.040

# Noise and model mismatch windows.
burst_start, burst_end = 28, 40
mismatch_start, mismatch_end = 50, 66

sigma_profile = np.full(n_blocks, 0.045)
sigma_profile[burst_start:burst_end] = 0.16
sigma_profile[mismatch_start:mismatch_end] = 0.095

distortion_profile = np.zeros(n_blocks)
distortion_profile[mismatch_start:mismatch_end] = 1.0
residual_distortion_scale = 0.045

Omega_true = target_Omega + true_delta_Omega
B_true = target_B + true_delta_B
A_true = target_A + 0.015 * np.sin(2 * np.pi * block_times / 23)

print(f"n_blocks = {n_blocks}")
print(f"burst window = {burst_start}:{burst_end}")
print(f"mismatch window = {mismatch_start}:{mismatch_end}")
print(f"shock window = {shock_start}:{shock_end}")


## Generate calibration data and fit modeled response

The fitting model is intentionally simpler than the simulated response, so model mismatch appears as structured error.


In [ ]:
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = distorted_rabi_model(
        pulse_t,
        A_true[k],
        Omega_true[k],
        B_true[k],
        distortion=distortion_profile[k] * residual_distortion_scale,
    )
    y_obs = np.clip(y_clean + sigma_profile[k] * np.random.randn(len(pulse_t)), 0, 1)
    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_obs = np.array(Y_obs)
Y_clean = np.array(Y_clean)

fit_params = []
fit_stds = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 6.0, 0.5])

for k in range(n_blocks):
    params, cov = fit_model(rabi_model, pulse_t, Y_obs[k], p0=initial_guess, bounds=bounds)
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)

delta_Omega_est = fit_params[:, 1] - target_Omega
delta_B_est = fit_params[:, 2] - target_B

print("Boundary calibration fits complete.")
print(f"measured Ω/B correlation = {np.corrcoef(delta_Omega_est, delta_B_est)[0,1]:.3f}")
print(f"median sigma Ω fit = {np.median(fit_stds[:,1]):.3e}")
print(f"median sigma B fit = {np.median(fit_stds[:,2]):.3e}")


## Define control policies


In [ ]:
delta_Omega_none = np.zeros(n_blocks)
delta_B_none = np.zeros(n_blocks)

delta_Omega_ma = moving_average(delta_Omega_est, window=4)
delta_B_ma = moving_average(delta_B_est, window=4)

# Clamp R so fixed Kalman is not unrealistically perfect.
r_omega = max(float(np.median(fit_stds[:, 1] ** 2)), 1e-5)
r_b = max(float(np.median(fit_stds[:, 2] ** 2)), 1e-6)

q_omega = 1.8e-2
q_b = 2.5e-4
q_cross = 0.35 * np.sqrt(q_omega * q_b)

delta_Omega_scalar, _, _, innov_omega = kalman_filter_1d(delta_Omega_est, q=q_omega, r=r_omega)
delta_B_scalar, _, _, innov_b = kalman_filter_1d(delta_B_est, q=q_b, r=r_b)

joint_fixed, joint_innov = joint_kalman(
    np.column_stack([delta_Omega_est, delta_B_est]),
    q_omega=q_omega,
    q_b=q_b,
    q_cross=q_cross,
    r_omega=r_omega,
    r_b=r_b,
)

delta_Omega_joint = joint_fixed[:, 0]
delta_B_joint = joint_fixed[:, 1]

robust_joint, robust_innov, gated_blocks = robust_gated_joint_kalman(
    np.column_stack([delta_Omega_est, delta_B_est]),
    q_omega=q_omega,
    q_b=q_b,
    q_cross=q_cross,
    r_omega=r_omega,
    r_b=r_b,
    gate_omega=0.20,
    gate_b=0.055,
)

delta_Omega_robust = robust_joint[:, 0]
delta_B_robust = robust_joint[:, 1]

delta_Omega_mpc = constrained_mpc_command(delta_Omega_joint, delta_bound=0.12, smooth=0.35)
delta_B_mpc = constrained_mpc_command(delta_B_joint, delta_bound=0.018, smooth=0.45)

delta_Omega_oracle = true_delta_Omega.copy()
delta_B_oracle = true_delta_B.copy()

print(f"r_omega = {r_omega:.3e}")
print(f"r_b = {r_b:.3e}")
print(f"q_omega = {q_omega:.3e}")
print(f"q_b = {q_b:.3e}")
print(f"q_cross = {q_cross:.3e}")
print(f"gated blocks = {np.where(gated_blocks)[0].tolist()}")


## Drift tracking under boundary pressure


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true Ω drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, alpha=0.7, label="measured Ω")
plt.plot(block_times, delta_Omega_ma, linestyle="--", label="moving average")
plt.plot(block_times, delta_Omega_scalar, linestyle=":", linewidth=2, label="scalar Kalman")
plt.plot(block_times, delta_Omega_joint, linestyle="-.", linewidth=2, label="joint Kalman")
plt.plot(block_times, delta_Omega_robust, linewidth=2, label="robust gated Kalman")
plt.plot(block_times, delta_Omega_mpc, linewidth=2, label="constrained MPC-lite")
plt.axvspan(burst_start, burst_end, alpha=0.12, label="noise burst")
plt.axvspan(mismatch_start, mismatch_end, alpha=0.08, label="model mismatch")
plt.xlabel("calibration block")
plt.ylabel("Ω drift / command")
plt.title("Failure boundary: Ω tracking and control command")
plt.legend()
plt.tight_layout()
save_fig("omega_tracking")
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true B drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, alpha=0.7, label="measured B")
plt.plot(block_times, delta_B_ma, linestyle="--", label="moving average")
plt.plot(block_times, delta_B_scalar, linestyle=":", linewidth=2, label="scalar Kalman")
plt.plot(block_times, delta_B_joint, linestyle="-.", linewidth=2, label="joint Kalman")
plt.plot(block_times, delta_B_robust, linewidth=2, label="robust gated Kalman")
plt.plot(block_times, delta_B_mpc, linewidth=2, label="constrained MPC-lite")
plt.axvspan(burst_start, burst_end, alpha=0.12, label="noise burst")
plt.axvspan(mismatch_start, mismatch_end, alpha=0.08, label="model mismatch")
plt.xlabel("calibration block")
plt.ylabel("B drift / command")
plt.title("Failure boundary: B tracking and control command")
plt.legend()
plt.tight_layout()
save_fig("offset_tracking")
plt.show()


## Evaluate phase-lock survival


In [ ]:
policies = {
    "none": (delta_Omega_none, delta_B_none),
    "moving_average": (delta_Omega_ma, delta_B_ma),
    "scalar_kalman": (delta_Omega_scalar, delta_B_scalar),
    "joint_kalman": (delta_Omega_joint, delta_B_joint),
    "robust_gated_kalman": (delta_Omega_robust, delta_B_robust),
    "constrained_mpc": (delta_Omega_mpc, delta_B_mpc),
    "oracle": (delta_Omega_oracle, delta_B_oracle),
}

policy_results = {}
for name, (om_hat, b_hat) in policies.items():
    policy_results[name] = evaluate_policy(name, om_hat, b_hat, true_delta_Omega, true_delta_B)

for name, result in policy_results.items():
    print(
        f"{name:22s} | "
        f"mean RMSE={result['response_rmse_mean']:.5f} | "
        f"max={result['response_rmse_max']:.5f} | "
        f"min cos={result['min_phase_lock']:.5f} | "
        f"failures={result['failure_count']}"
    )


## Response RMSE comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    plt.plot(block_times, result["response_rmse"], marker="o", linewidth=1, label=name)

plt.axvspan(burst_start, burst_end, alpha=0.12, label="noise burst")
plt.axvspan(mismatch_start, mismatch_end, alpha=0.08, label="model mismatch")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.title("Failure boundary: response RMSE comparison")
plt.legend()
plt.tight_layout()
save_fig("response_rmse_comparison")
plt.show()


## CGCS phase-lock stability


In [ ]:
threshold = 1 / np.sqrt(2)

plt.figure(figsize=(11, 5))
plt.axhline(threshold, linestyle="--", linewidth=1, label="45° threshold")

for name, result in policy_results.items():
    plt.plot(block_times, result["phase_lock"], marker="o", linewidth=1, label=name)

plt.axvspan(burst_start, burst_end, alpha=0.12, label="noise burst")
plt.axvspan(mismatch_start, mismatch_end, alpha=0.08, label="model mismatch")
plt.ylim(0.55, 1.01)
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.title("Failure boundary: CGCS phase-lock stability")
plt.legend()
plt.tight_layout()
save_fig("cgcs_stability_comparison")
plt.show()


## Threshold margin

Margin:

```text
margin = cos θ − 1 / √(1² + 1²)
```

A negative margin means phase-lock failure.


In [ ]:
plt.figure(figsize=(11, 5))
plt.axhline(0.0, linestyle="--", linewidth=1, label="threshold margin = 0")

for name, result in policy_results.items():
    if name == "oracle":
        continue
    plt.plot(block_times, result["phase_lock"] - threshold, marker="o", linewidth=1, label=name)

plt.axvspan(burst_start, burst_end, alpha=0.12, label="noise burst")
plt.axvspan(mismatch_start, mismatch_end, alpha=0.08, label="model mismatch")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.title("Failure boundary: threshold margin")
plt.legend()
plt.tight_layout()
save_fig("threshold_margin")
plt.show()


## Failure map


In [ ]:
failure_map = np.array([
    policy_results[name]["phase_lock"] < threshold
    for name in policy_results.keys()
], dtype=int)

plt.figure(figsize=(11, 3.8))
plt.imshow(failure_map, aspect="auto", interpolation="nearest")
plt.yticks(np.arange(len(policy_results)), list(policy_results.keys()))
plt.xticks(np.arange(0, n_blocks, 5))
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.title("Failure boundary: phase-lock failure map")
plt.colorbar(label="failure: cos θ < threshold")
plt.tight_layout()
save_fig("failure_map")
plt.show()


## Policy ranking


In [ ]:
ranking = sorted(policy_results.values(), key=lambda r: (r["failure_count"], r["response_rmse_mean"]))

policy_table = []

for rank, result in enumerate(ranking, start=1):
    policy_table.append({
        "rank": rank,
        "policy": result["name"],
        "omega_rmse": result["omega_rmse"],
        "b_rmse": result["b_rmse"],
        "response_rmse_mean": result["response_rmse_mean"],
        "response_rmse_max": result["response_rmse_max"],
        "min_phase_lock": result["min_phase_lock"],
        "mean_phase_lock": result["mean_phase_lock"],
        "max_angle_degrees": result["max_angle_degrees"],
        "all_blocks_phase_locked": result["all_blocks_phase_locked"],
        "failure_count": result["failure_count"],
        "first_failure_block": result["first_failure_block"],
    })

policy_table


In [ ]:
names = [r["policy"] for r in policy_table]
means = [r["response_rmse_mean"] for r in policy_table]
maxes = [r["response_rmse_max"] for r in policy_table]

x = np.arange(len(names))
width = 0.35

plt.figure(figsize=(11, 5))
plt.bar(x - width/2, means, width, label="mean RMSE")
plt.bar(x + width/2, maxes, width, label="max RMSE")
plt.xticks(x, names, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Failure boundary: policy ranking")
plt.legend()
plt.tight_layout()
save_fig("policy_ranking_summary")
plt.show()


## Gate sweep

Stress-test robust Kalman gate threshold.

Smaller gate rejects more observations. Larger gate behaves like ordinary joint Kalman.


In [ ]:
gate_values = [0.06, 0.10, 0.14, 0.18, 0.22, 0.30, 0.42]
gate_mean_rmse = []
gate_failures = []

for gate in gate_values:
    robust_tmp, _, _ = robust_gated_joint_kalman(
        np.column_stack([delta_Omega_est, delta_B_est]),
        q_omega=q_omega,
        q_b=q_b,
        q_cross=q_cross,
        r_omega=r_omega,
        r_b=r_b,
        gate_omega=gate,
        gate_b=0.35 * gate,
    )

    ev = evaluate_policy(
        f"gate_{gate}",
        robust_tmp[:, 0],
        robust_tmp[:, 1],
        true_delta_Omega,
        true_delta_B,
    )

    gate_mean_rmse.append(ev["response_rmse_mean"])
    gate_failures.append(ev["failure_count"])

gate_mean_rmse = np.array(gate_mean_rmse)
gate_failures = np.array(gate_failures)
best_gate_idx = int(np.argmin(gate_mean_rmse))
best_gate = float(gate_values[best_gate_idx])

print(f"best gate = {best_gate:.3f}")
print(f"best gate RMSE = {gate_mean_rmse[best_gate_idx]:.6f}")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(gate_values, gate_mean_rmse, marker="o", label="mean RMSE")
plt.axvline(best_gate, linestyle=":", label=f"best gate = {best_gate:.2f}")
plt.xlabel("Ω outlier gate threshold")
plt.ylabel("mean response RMSE")
plt.title("Failure boundary: robust gate sweep")
plt.legend()
plt.tight_layout()
save_fig("robust_gate_sweep")
plt.show()


## Boundary strength sweep

Increase drift strength and measure first phase-lock failure.


In [ ]:
strengths = np.linspace(0.6, 1.8, 9)
strength_results = []

base_omega = true_delta_Omega.copy()
base_b = true_delta_B.copy()

for s in strengths:
    scaled_omega = s * base_omega
    scaled_b = s * base_b

    result_none = evaluate_policy("none", np.zeros(n_blocks), np.zeros(n_blocks), scaled_omega, scaled_b)
    result_joint = evaluate_policy("joint", delta_Omega_joint, delta_B_joint, scaled_omega, scaled_b)
    result_mpc = evaluate_policy("constrained_mpc", delta_Omega_mpc, delta_B_mpc, scaled_omega, scaled_b)

    strength_results.append({
        "strength": float(s),
        "none_min_cos": result_none["min_phase_lock"],
        "joint_min_cos": result_joint["min_phase_lock"],
        "mpc_min_cos": result_mpc["min_phase_lock"],
        "none_failures": result_none["failure_count"],
        "joint_failures": result_joint["failure_count"],
        "mpc_failures": result_mpc["failure_count"],
    })

strength_results


In [ ]:
plt.figure(figsize=(9, 5))
plt.axhline(threshold, linestyle="--", label="45° threshold")
plt.plot(strengths, [r["none_min_cos"] for r in strength_results], marker="o", label="none")
plt.plot(strengths, [r["joint_min_cos"] for r in strength_results], marker="o", label="joint_kalman")
plt.plot(strengths, [r["mpc_min_cos"] for r in strength_results], marker="o", label="constrained_mpc")
plt.xlabel("drift strength multiplier")
plt.ylabel("minimum cosine similarity")
plt.title("Failure boundary: drift-strength sweep")
plt.legend()
plt.tight_layout()
save_fig("drift_strength_sweep")
plt.show()


## Worst-case block comparison


In [ ]:
worst_block = int(np.argmax(policy_results["none"]["response_rmse"]))

plt.figure(figsize=(11, 5))
plt.plot(pulse_t, target_signal, linewidth=3, label="target")

for name in ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "constrained_mpc", "oracle"]:
    plt.plot(
        pulse_t,
        policy_results[name]["controlled_signals"][worst_block],
        linewidth=2,
        label=name,
    )

plt.xlabel("pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Failure boundary: worst-case block {worst_block}")
plt.legend()
plt.tight_layout()
save_fig("worst_case_block_comparison")
plt.show()


## Save summary JSON


In [ ]:
summary = {
    "notebook": "12_failure_boundary_phase_lock",
    "blocks": int(n_blocks),
    "phase_lock_threshold": float(threshold),
    "windows": {
        "noise_burst": [int(burst_start), int(burst_end)],
        "model_mismatch": [int(mismatch_start), int(mismatch_end)],
        "coupling_shock": [int(shock_start), int(shock_end)],
    },
    "covariance": {
        "q_omega": float(q_omega),
        "q_b": float(q_b),
        "q_cross": float(q_cross),
        "r_omega": float(r_omega),
        "r_b": float(r_b),
    },
    "robust_gate": {
        "current_gate_omega": 0.20,
        "current_gate_b": 0.055,
        "best_gate_omega": float(best_gate),
        "best_gate_response_rmse": float(gate_mean_rmse[best_gate_idx]),
        "gated_blocks": [int(x) for x in np.where(gated_blocks)[0].tolist()],
    },
    "ranking": policy_table,
    "strength_sweep": strength_results,
}

with open(f"{RESULTS_DIR}/failure_boundary_phase_lock_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/failure_boundary_phase_lock_summary.json")


## Write docs and results markdown


In [ ]:
docs_md = '''# Failure Boundary Phase-Lock

Notebook 12 stress-tests control policies near the CGCS 45° phase-lock boundary.

---

## Purpose

Previous notebooks mostly measured precision inside a stable phase-lock regime.

This notebook intentionally increases:

- Ω drift
- B drift
- measurement burst noise
- model mismatch
- coupling shock

The goal is to identify which control policies preserve phase-lock and which fail.

---

## Figures

### Ω tracking

![Omega tracking](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_omega_tracking.png)

Large Ω drift creates boundary pressure for uncontrolled and lagged policies.

---

### B tracking

![Offset tracking](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_offset_tracking.png)

Offset drift and coupling shock test whether controllers stabilize readout bias.

---

### Response RMSE comparison

![Response RMSE](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_response_rmse_comparison.png)

Response-level RMSE shows boundary-pressure error across policies.

---

### CGCS phase-lock stability

![CGCS stability](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_cgcs_stability_comparison.png)

Phase-lock stability is measured against the 45° threshold.

---

### Threshold margin

![Threshold margin](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_threshold_margin.png)

Margin is cosine similarity minus the CGCS threshold.

---

### Failure map

![Failure map](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_failure_map.png)

The failure map marks blocks where cosθ falls below the 45° threshold.

---

### Policy ranking

![Policy ranking](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_policy_ranking_summary.png)

Policy ranking sorts by failure count and mean response RMSE.

---

### Robust gate sweep

![Gate sweep](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_robust_gate_sweep.png)

Gate sweep tests how aggressively the robust Kalman filter rejects outlier observations.

---

### Drift-strength sweep

![Strength sweep](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_drift_strength_sweep.png)

Drift-strength sweep identifies where policies approach or cross the phase-lock boundary.

---

### Worst-case block comparison

![Worst case](../figures/failure_boundary_phase_lock/12_failure_boundary_phase_lock_worst_case_block_comparison.png)

Worst-case block shows response-shape deviation under maximum boundary pressure.

---

## Key Takeaway

Notebook 12 shifts the control stack from precision comparison to failure-boundary testing.

The important metric is no longer only RMSE.

The new metric is:

```text
survival margin = cosθ − 1 / √(1² + 1²)
```

Negative margin indicates phase-lock failure.
'''

summary_rows = "\n".join(
    [
        f"| {r['rank']} | {r['policy']} | {r['failure_count']} | {r['response_rmse_mean']:.6f} | {r['response_rmse_max']:.6f} | {r['min_phase_lock']:.6f} |"
        for r in policy_table
    ]
)

summary_md = f'''# Failure Boundary Phase-Lock Results Summary

---

## Configuration

- Blocks: {n_blocks}
- Phase-lock threshold: {threshold:.4f}
- Noise burst window: {burst_start}–{burst_end}
- Model mismatch window: {mismatch_start}–{mismatch_end}
- Coupling shock window: {shock_start}–{shock_end}

---

## Covariance

- q_omega: {q_omega:.3e}
- q_b: {q_b:.3e}
- q_cross: {q_cross:.3e}
- r_omega: {r_omega:.3e}
- r_b: {r_b:.3e}

---

## Robust Gate

- Current Ω gate: 0.20
- Current B gate: 0.055
- Best Ω gate: {best_gate:.3f}
- Best gate RMSE: {gate_mean_rmse[best_gate_idx]:.6f}
- Gated blocks: {np.where(gated_blocks)[0].tolist()}

---

## Policy Ranking

| Rank | Policy | Failures | Mean RMSE | Max RMSE | Min Cosine |
|------|--------|---------:|----------:|---------:|-----------:|
{summary_rows}

---

## Key Interpretation

Notebook 12 tests whether controllers preserve CGCS phase-lock under large drift, noise bursts, model mismatch, and coupling shock.

A policy fails when:

```text
cosθ < 1 / √(1² + 1²)
```

The strongest practical controller is the one with the fewest phase-lock failures and lowest response RMSE.
'''

with open(f"{DOCS_DIR}/failure_boundary_phase_lock.md", "w") as f:
    f.write(docs_md)

with open(f"{RESULTS_DIR}/failure_boundary_phase_lock_summary.md", "w") as f:
    f.write(summary_md)

print(f"Wrote {DOCS_DIR}/failure_boundary_phase_lock.md")
print(f"Wrote {RESULTS_DIR}/failure_boundary_phase_lock_summary.md")


## Zip outputs for download from Colab


In [ ]:
!zip -r failure_boundary_phase_lock_outputs.zip figures results docs


## Control-stack status

Notebook 12 adds phase-lock failure-boundary testing.

Next:

```text
13_recovery_time_and_resilience.ipynb
```

Goal:

> measure how quickly each controller returns to phase-lock after a burst, shock, or model-mismatch event.
